## Run CELL 1,2 Everytime when you restart

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import json
import torch

# Load text
with open('/content/drive/MyDrive/mini_gpt_marathi/marathi_text.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# Load vocabulary
with open('/content/drive/MyDrive/mini_gpt_marathi/vocab.json', 'r', encoding='utf-8') as f:
    vocab = json.load(f)

chars = vocab['chars']
char_to_idx = vocab['char_to_idx']
idx_to_char = {int(k): v for k, v in vocab['idx_to_char'].items()}
vocab_size = vocab['vocab_size']

# Recreate encode decode functions
def encode(text):
    return [char_to_idx[ch] for ch in text]

def decode(indices):
    return "".join([idx_to_char[idx] for idx in indices])

print("Data loaded from Drive ✅")
print(f"Text length — {len(text)}")
print(f"Vocab size — {vocab_size}")

Mounted at /content/drive
Data loaded from Drive ✅
Text length — 2089215
Vocab size — 222


In [2]:
# Encode and split data
data = torch.tensor(encode(text), dtype=torch.long)
split = int(0.9 * len(data))
train_data = data[:split]
val_data = data[split:]

device = "cuda" if torch.cuda.is_available() else "cpu"
train_data = train_data.to(device)
val_data = val_data.to(device)

print(f"Device — {device}")
print(f"Train tokens — {len(train_data)}")
print(f"Val tokens — {len(val_data)}")

Device — cuda
Train tokens — 1880293
Val tokens — 208922


In [ ]:
## Check GPU

In [ ]:
import torch

# Check if GPU is available
print("GPU Available:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")
print("PyTorch Version:", torch.__version__)

GPU Available: True
GPU Name: Tesla T4
PyTorch Version: 2.11.0+cu130


## Load Dataset


In [4]:
from datasets import load_dataset

print("Downloading Marathi dataset...")

dataset = load_dataset(
    "ai4bharat/sangraha",
    "verified",
    trust_remote_code=True,
    split="mar",  # mar = Marathi language code
    streaming=True
)

print("Dataset loaded!")

# Collect Marathi texts
count = 0
marathi_texts = []

for example in dataset:
    marathi_texts.append(example["text"])
    count += 1

    if count == 1000:  # get 1000 examples
        break

print(f"Collected {len(marathi_texts)} Marathi texts")
print("\nSample Marathi text —")
print(marathi_texts[0][:500])

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'ai4bharat/sangraha' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'ai4bharat/sangraha' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


README.md:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/77 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/50 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/27 [00:00<?, ?it/s]

...

Resolving data files:   0%|          | 0/25 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/53 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/27 [00:00<?, ?it/s]

Dataset loaded!
Collected 1000 Marathi texts

Sample Marathi text —
शासनाने प्रादेशिक योजनेतील गावांना पाणीपुरवठा होत नसल्याने विद्युत व तांत्रिक प्रस्ताव मागविले असून यापूर्वीच्या रखडलेल्या वीजदेयकांची अदायगी करण्याचा विचार करीत आहे. हिंगोली जि. प. ने सिद्धेश्वर, पुरजळ व मोरवाडी योजनेचे प्रस्ताव यात पाठविल्याची माहिती मुख्य कार्यकारी अधिकारी एच. पी. तुम्मोड यांनी दिली.
 हिंगोली : शासनाने प्रादेशिक योजनेतील गावांना पाणीपुरवठा होत नसल्याने विद्युत व तांत्रिक प्रस्ताव मागविले असून यापूर्वीच्या रखडलेल्या वीजदेयकांची अदायगी करण्याचा विचार करीत आहे. हिंगोली जि. प. ने


In [ ]:
## Prepare and Save the data

In [5]:
# Combine all texts into one big text
print("Preparing data...")

# Join all texts with newline separator
full_text = "\n\n".join(marathi_texts)

print(f"Total characters — {len(full_text)}")
print(f"Total words — {len(full_text.split())}")
print("\nFirst 500 characters —")
print(full_text[:500])

# Save to file
with open("marathi_text.txt", "w", encoding="utf-8") as f:
    f.write(full_text)

print("\nData saved to marathi_text.txt ✅")

Preparing data...
Total characters — 2089215
Total words — 303556

First 500 characters —
शासनाने प्रादेशिक योजनेतील गावांना पाणीपुरवठा होत नसल्याने विद्युत व तांत्रिक प्रस्ताव मागविले असून यापूर्वीच्या रखडलेल्या वीजदेयकांची अदायगी करण्याचा विचार करीत आहे. हिंगोली जि. प. ने सिद्धेश्वर, पुरजळ व मोरवाडी योजनेचे प्रस्ताव यात पाठविल्याची माहिती मुख्य कार्यकारी अधिकारी एच. पी. तुम्मोड यांनी दिली.
 हिंगोली : शासनाने प्रादेशिक योजनेतील गावांना पाणीपुरवठा होत नसल्याने विद्युत व तांत्रिक प्रस्ताव मागविले असून यापूर्वीच्या रखडलेल्या वीजदेयकांची अदायगी करण्याचा विचार करीत आहे. हिंगोली जि. प. ने

Data saved to marathi_text.txt ✅


## Build Tokenizer for Marathi

In [6]:
# Read the saved text
with open("marathi_text.txt", "r", encoding="utf-8") as f:
    text = f.read()

print(f"Total text length — {len(text)} characters")

# Step 1 — Find all unique characters
chars = sorted(list(set(text)))
vocab_size = len(chars)

print(f"Total unique characters — {vocab_size}")
print(f"\nFirst 50 characters in vocabulary —")
print(chars[:50])

# Step 2 — Create character to number mapping
char_to_idx = {ch: idx for idx, ch in enumerate(chars)}

# Step 3 — Create number to character mapping
idx_to_char = {idx: ch for idx, ch in enumerate(chars)}

print("\nSample mappings —")
print(f"'म' maps to — {char_to_idx.get('म', 'not found')}")
print(f"'र' maps to — {char_to_idx.get('र', 'not found')}")
print(f"'ा' maps to — {char_to_idx.get('ा', 'not found')}")

# Step 4 — Create encode and decode functions
def encode(text):
    # Convert text to list of numbers
    return [char_to_idx[ch] for ch in text]

def decode(indices):
    # Convert list of numbers back to text
    return "".join([idx_to_char[idx] for idx in indices])

# Test encode and decode
sample = "मराठी भाषा"
encoded = encode(sample)
decoded = decode(encoded)

print(f"\nOriginal  — {sample}")
print(f"Encoded   — {encoded}")
print(f"Decoded   — {decoded}")
print(f"Match     — {sample == decoded}")

Total text length — 2089215 characters
Total unique characters — 222

First 50 characters in vocabulary —
['\n', ' ', '!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '=', '>', '?', '@', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q']

Sample mappings —
'म' maps to — 168
'र' maps to — 170
'ा' maps to — 181

Original  — मराठी भाषा
Encoded   — [168, 170, 181, 155, 183, 1, 167, 181, 176, 181]
Decoded   — मराठी भाषा
Match     — True


## Prepare Training Data

In [7]:
import torch

# Step 1 — Encode entire text to numbers
print("Encoding entire text...")
data = torch.tensor(encode(text), dtype=torch.long)
print(f"Total tokens — {len(data)}")
print(f"Data type — {data.dtype}")
print(f"First 20 tokens — {data[:20]}")

# Step 2 — Split into train and validation
split = int(0.9 * len(data))  # 90% train, 10% validation

train_data = data[:split]
val_data = data[split:]

print(f"\nTrain tokens — {len(train_data)}")
print(f"Validation tokens — {len(val_data)}")

# Step 3 — Understand how GPT reads data
# GPT reads text in chunks called block_size
block_size = 128  # GPT reads 128 characters at a time

# Example of how GPT learns
print(f"\nHow GPT learns from data —")
example = train_data[:block_size+1]
print(f"Input sequence — {example[:-1]}")
print(f"Target sequence — {example[1:]}")

print("\nIn simple words —")
for i in range(5):
    context = example[:i+1]
    target = example[i+1]
    print(f"Input {decode(context.tolist())} → Predict {decode([target.item()])}")

Encoding entire text...
Total tokens — 2089215
Data type — torch.int64
First 20 tokens — tensor([175, 181, 177, 163, 181, 163, 189,   1, 164, 194, 170, 181, 161, 189,
        175, 182, 144,   1, 169, 192])

Train tokens — 1880293
Validation tokens — 208922

How GPT learns from data —
Input sequence — tensor([175, 181, 177, 163, 181, 163, 189,   1, 164, 194, 170, 181, 161, 189,
        175, 182, 144,   1, 169, 192, 151, 163, 189, 159, 183, 172,   1, 146,
        181, 174, 181, 129, 163, 181,   1, 164, 181, 158, 183, 164, 184, 170,
        174, 155, 181,   1, 178, 192, 159,   1, 163, 177, 172, 194, 169, 181,
        163, 189,   1, 174, 182, 161, 194, 169, 184, 159,   1, 174,   1, 159,
        181, 129, 159, 194, 170, 182, 144,   1, 164, 194, 170, 177, 194, 159,
        181, 174,   1, 168, 181, 146, 174, 182, 172, 189,   1, 131, 177, 185,
        163,   1, 169, 181, 164, 185, 170, 194, 174, 183, 149, 194, 169, 181,
          1, 170, 145, 156, 172, 189, 172, 194, 169, 181,   1, 174, 183, 1

## Create Data Loader

In [8]:
# Hyperparameters
# These are settings that control how model trains

batch_size = 32      # how many chunks to train on at once
block_size = 128     # how many characters in each chunk
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using device — {device}")
print(f"Batch size — {batch_size}")
print(f"Block size — {block_size}")

# Move data to GPU
train_data = train_data.to(device)
val_data = val_data.to(device)

def get_batch(split):
    """
    Get random batch of data for training

    What happens here —
    1. Pick random starting positions in text
    2. Extract chunk of block_size characters from each position
    3. Target is same chunk shifted by 1 — next character prediction
    4. Return inputs and targets as tensors
    """
    # Choose train or validation data
    data = train_data if split == "train" else val_data

    # Pick random starting positions
    # Each position is start of one training example
    ix = torch.randint(len(data) - block_size, (batch_size,))

    # Extract chunks
    x = torch.stack([data[i:i+block_size] for i in ix])

    # Target is same chunk shifted by 1
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])

    return x, y

# Test get_batch
x, y = get_batch("train")
print(f"\nBatch shapes —")
print(f"Input shape  — {x.shape}")
print(f"Target shape — {y.shape}")
print(f"\nThis means —")
print(f"{batch_size} examples, each with {block_size} characters")

Using device — cuda
Batch size — 32
Block size — 128

Batch shapes —
Input shape  — torch.Size([32, 128])
Target shape — torch.Size([32, 128])

This means —
32 examples, each with 128 characters


## Save dataset to Google Drive

In [9]:
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Create project folder in Drive
import os
os.makedirs('/content/drive/MyDrive/mini_gpt_marathi', exist_ok=True)

# Save text data to Drive
with open('/content/drive/MyDrive/mini_gpt_marathi/marathi_text.txt', 'w', encoding='utf-8') as f:
    f.write(full_text)

# Save vocabulary
import json
vocab = {
    'chars': chars,
    'char_to_idx': char_to_idx,
    'idx_to_char': {str(k): v for k, v in idx_to_char.items()},
    'vocab_size': vocab_size
}

with open('/content/drive/MyDrive/mini_gpt_marathi/vocab.json', 'w', encoding='utf-8') as f:
    json.dump(vocab, f, ensure_ascii=False)

print("Dataset saved to Google Drive ✅")
print("Vocabulary saved to Google Drive ✅")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset saved to Google Drive ✅
Vocabulary saved to Google Drive ✅


## Build Self Attention Head

In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Hyperparameters — settings that control model size
batch_size   = 32    # examples per batch
block_size   = 128   # characters per example
n_embd       = 256   # embedding dimension — size of each word vector
n_head       = 8     # number of attention heads
n_layer      = 6     # number of transformer blocks
dropout      = 0.2   # dropout rate — prevents overfitting
learning_rate = 3e-4 # how fast model learns

print("Hyperparameters set ✅")
print(f"Embedding size  — {n_embd}")
print(f"Attention heads — {n_head}")
print(f"Layers          — {n_layer}")
print(f"Block size      — {block_size}")
print(f"Vocab size      — {vocab_size}")

class SingleHead(nn.Module):
    """
    Single Self Attention Head

    What happens here —
    1. Each character gets Q K V vectors
    2. Q × K gives attention scores
    3. Scores scaled and masked
    4. Softmax gives attention weights
    5. Weights × V gives output
    """
    def __init__(self, head_size):
        super().__init__()

        # Q K V linear layers
        # No bias needed in attention
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        # Mask — prevents looking at future characters
        # GPT can only look at past characters not future
        self.register_buffer(
            'tril',
            torch.tril(torch.ones(block_size, block_size))
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape  # Batch, Time, Channels

        # Step 1 — Create Q K V
        q = self.query(x)  # (B, T, head_size)
        k = self.key(x)    # (B, T, head_size)
        v = self.value(x)  # (B, T, head_size)

        # Step 2 — Calculate attention scores
        # Scale by sqrt of head_size for stability
        head_size = k.shape[-1]
        scores = q @ k.transpose(-2, -1) * (head_size ** -0.5)
        # scores shape — (B, T, T)

        # Step 3 — Mask future positions
        # GPT cannot see future characters
        scores = scores.masked_fill(
            self.tril[:T, :T] == 0,
            float('-inf')
        )

        # Step 4 — Softmax to get weights
        weights = F.softmax(scores, dim=-1)
        weights = self.dropout(weights)

        # Step 5 — Weighted sum of values
        out = weights @ v  # (B, T, head_size)

        return out

# Test single head
head_size = n_embd // n_head  # 256 // 8 = 32
head = SingleHead(head_size).to(device)

# Dummy input
test_input = torch.randn(batch_size, block_size, n_embd).to(device)
test_output = head(test_input)

print(f"\nSingle Head Test —")
print(f"Input shape  — {test_input.shape}")
print(f"Output shape — {test_output.shape}")
print(f"Head size    — {head_size}")

Hyperparameters set ✅
Embedding size  — 256
Attention heads — 8
Layers          — 6
Block size      — 128
Vocab size      — 222

Single Head Test —
Input shape  — torch.Size([32, 128, 256])
Output shape — torch.Size([32, 128, 32])
Head size    — 32


## Build Multi Head Attention

In [11]:
class MultiHeadAttention(nn.Module):
    """
    Multiple Self Attention Heads running in parallel

    What happens here —
    1. Create n_head single attention heads
    2. Run all heads simultaneously on same input
    3. Concatenate all head outputs
    4. Project back to n_embd dimensions
    """
    def __init__(self, num_heads, head_size):
        super().__init__()

        # Create multiple heads
        self.heads = nn.ModuleList([
            SingleHead(head_size) for _ in range(num_heads)
        ])

        # Final linear layer to combine all heads
        # Input — num_heads * head_size = 8 * 32 = 256
        # Output — n_embd = 256
        self.proj = nn.Linear(n_embd, n_embd)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # Run all heads in parallel
        # Each head output — (B, T, head_size)
        head_outputs = [head(x) for head in self.heads]

        # Concatenate all head outputs
        # (B, T, head_size) × 8 → (B, T, 256)
        out = torch.cat(head_outputs, dim=-1)

        # Project back to n_embd
        out = self.proj(out)
        out = self.dropout(out)

        return out

# Test Multi Head Attention
mha = MultiHeadAttention(n_head, head_size).to(device)

test_output = mha(test_input)

print("Multi Head Attention Test —")
print(f"Input shape  — {test_input.shape}")
print(f"Output shape — {test_output.shape}")
print(f"Number of heads — {n_head}")
print(f"Head size       — {head_size}")
print(f"\nInput and output same shape ✅")
print(f"Each character now has info from all 8 attention heads")

Multi Head Attention Test —
Input shape  — torch.Size([32, 128, 256])
Output shape — torch.Size([32, 128, 256])
Number of heads — 8
Head size       — 32

Input and output same shape ✅
Each character now has info from all 8 attention heads


## Build Feed Forward Network

In [12]:
class FeedForward(nn.Module):
    """
    Feed Forward Network

    What happens here —
    1. Expand dimensions by 4x — 256 to 1024
    2. Apply ReLU activation — adds non linearity
    3. Compress back — 1024 to 256
    4. Apply dropout

    Why 4x expansion —
    More capacity to learn complex patterns
    Standard in all Transformer architectures
    """
    def __init__(self, n_embd):
        super().__init__()

        self.net = nn.Sequential(
            # Expand 256 → 1024
            nn.Linear(n_embd, 4 * n_embd),

            # ReLU activation
            # Adds non linearity — model learns complex patterns
            nn.ReLU(),

            # Compress 1024 → 256
            nn.Linear(4 * n_embd, n_embd),

            # Dropout — prevents overfitting
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)

# Test Feed Forward
ff = FeedForward(n_embd).to(device)

test_output = ff(test_input)

print("Feed Forward Network Test —")
print(f"Input shape  — {test_input.shape}")
print(f"Output shape — {test_output.shape}")
print(f"\nExpansion inside — {n_embd} → {4*n_embd} → {n_embd}")
print(f"Input and output same shape ✅")

Feed Forward Network Test —
Input shape  — torch.Size([32, 128, 256])
Output shape — torch.Size([32, 128, 256])

Expansion inside — 256 → 1024 → 256
Input and output same shape ✅


## Build Transformer Block

In [13]:
class TransformerBlock(nn.Module):
    """
    Complete Transformer Block

    What happens here —
    1. Layer Norm → Multi Head Attention → Residual Connection
    2. Layer Norm → Feed Forward Network → Residual Connection

    Why Layer Norm before attention —
    More stable than putting it after
    Called Pre-LN Transformer — modern standard

    Why Residual Connection —
    x = x + attention(x)
    x = x + feedforward(x)
    Input added to output — gradient flows easily
    """
    def __init__(self, n_embd, n_head):
        super().__init__()

        head_size = n_embd // n_head

        # Multi Head Attention
        self.attention = MultiHeadAttention(n_head, head_size)

        # Feed Forward Network
        self.ff = FeedForward(n_embd)

        # Layer Normalization
        # Applied before attention and feed forward
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        # Step 1 — Attention with Residual Connection
        # x = x + attention(layernorm(x))
        x = x + self.attention(self.ln1(x))

        # Step 2 — Feed Forward with Residual Connection
        # x = x + feedforward(layernorm(x))
        x = x + self.ff(self.ln2(x))

        return x

# Test Transformer Block
block = TransformerBlock(n_embd, n_head).to(device)

test_output = block(test_input)

print("Transformer Block Test —")
print(f"Input shape  — {test_input.shape}")
print(f"Output shape — {test_output.shape}")
print(f"\nInside one block —")
print(f"LayerNorm → MultiHeadAttention → Residual")
print(f"LayerNorm → FeedForward → Residual")
print(f"\nInput and output same shape ✅")

Transformer Block Test —
Input shape  — torch.Size([32, 128, 256])
Output shape — torch.Size([32, 128, 256])

Inside one block —
LayerNorm → MultiHeadAttention → Residual
LayerNorm → FeedForward → Residual

Input and output same shape ✅


## Build Complete Mini GPT Model

In [15]:
class MiniGPT(nn.Module):
    """
    Complete Mini GPT Model for Marathi Text Generation

    Components —
    1. Token Embedding    — converts character index to vector
    2. Position Embedding — adds position information
    3. Transformer Blocks — 6 blocks of attention + feedforward
    4. Layer Norm         — final normalization
    5. Linear Head        — predicts next character probabilities
    """
    def __init__(self):
        super().__init__()

        # Token Embedding
        # Converts character index to 256 dimensional vector
        # vocab_size = 222 unique Marathi characters
        self.token_embedding = nn.Embedding(vocab_size, n_embd)

        # Position Embedding
        # Each position gets its own learned vector
        # Without this — model has no sense of word order
        self.position_embedding = nn.Embedding(block_size, n_embd)

        # Stack of Transformer Blocks
        # 6 blocks stacked on top of each other
        self.blocks = nn.Sequential(
            *[TransformerBlock(n_embd, n_head) for _ in range(n_layer)]
        )

        # Final Layer Normalization
        self.ln_final = nn.LayerNorm(n_embd)

        # Linear Head — predicts next character
        # Takes 256 dimensional vector
        # Outputs probability for each of 222 characters
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # Step 1 — Token Embedding
        # idx — (B, T) → tok_emb — (B, T, 256)
        tok_emb = self.token_embedding(idx)

        # Step 2 — Position Embedding
        # positions — (T,) → pos_emb — (T, 256)
        positions = torch.arange(T, device=device)
        pos_emb = self.position_embedding(positions)

        # Step 3 — Add token and position embeddings
        # Both (B, T, 256) — added together
        x = tok_emb + pos_emb

        # Step 4 — Pass through 6 Transformer Blocks
        x = self.blocks(x)

        # Step 5 — Final Layer Norm
        x = self.ln_final(x)

        # Step 6 — Predict next character
        # (B, T, 256) → (B, T, 222)
        logits = self.lm_head(x)

        # Calculate loss if targets provided
        loss = None
        if targets is not None:
            # Reshape for cross entropy loss
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)

            # Cross Entropy Loss
            # Measures how wrong our predictions are
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        """
        Generate new Marathi text

        What happens —
        1. Take existing characters as context
        2. Predict next character
        3. Add predicted character to context
        4. Repeat max_new_tokens times
        """
        for _ in range(max_new_tokens):
            # Crop to block_size — cannot exceed context window
            idx_crop = idx[:, -block_size:]

            # Get predictions
            logits, _ = self(idx_crop)

            # Focus on last character prediction
            logits = logits[:, -1, :]  # (B, vocab_size)

            # Convert to probabilities
            probs = F.softmax(logits, dim=-1)

            # Sample next character
            idx_next = torch.multinomial(probs, num_samples=1)

            # Add to sequence
            idx = torch.cat([idx, idx_next], dim=1)

        return idx

# Create model
model = MiniGPT().to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print("Mini GPT Model Created! ✅")
print(f"\nModel Architecture —")
print(f"Vocab size        — {vocab_size}")
print(f"Embedding size    — {n_embd}")
print(f"Attention heads   — {n_head}")
print(f"Transformer layers— {n_layer}")
print(f"Block size        — {block_size}")
print(f"\nTotal parameters  — {total_params:,}")
print(f"Total parameters  — {total_params/1e6:.2f} Million")

# Test forward pass
x_test, y_test = get_batch("train")
logits, loss = model(x_test, y_test)

print(f"\nForward Pass Test —")
print(f"Input shape   — {x_test.shape}")
print(f"Logits shape  — {logits.shape}")
print(f"Initial loss  — {loss.item():.4f}")
print(f"\nExpected initial loss — {torch.log(torch.tensor(vocab_size)):.4f}")
print(f"(Random prediction loss for {vocab_size} characters)")

Mini GPT Model Created! ✅

Model Architecture —
Vocab size        — 222
Embedding size    — 256
Attention heads   — 8
Transformer layers— 6
Block size        — 128

Total parameters  — 4,881,118
Total parameters  — 4.88 Million

Forward Pass Test —
Input shape   — torch.Size([32, 128])
Logits shape  — torch.Size([4096, 222])
Initial loss  — 5.6173

Expected initial loss — 5.4027
(Random prediction loss for 222 characters)


## Train the Model

In [16]:
# Training settings
max_iters   = 5000   # total training steps
eval_every  = 500    # evaluate every 500 steps
eval_iters  = 50     # use 50 batches for evaluation

# Optimizer
# AdamW — best optimizer for Transformer training
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate
)

print("Starting Mini GPT Training on Marathi Text...")
print(f"Total steps     — {max_iters}")
print(f"Evaluate every  — {eval_every} steps")
print(f"Learning rate   — {learning_rate}")
print(f"Device          — {device}")
print("-" * 50)

@torch.no_grad()
def estimate_loss():
    """
    Estimate loss on train and validation data

    Why @torch.no_grad() —
    During evaluation we don't need gradients
    Saves memory and speeds up evaluation
    """
    model.eval()  # set model to evaluation mode
    losses = {}

    for split in ["train", "val"]:
        split_losses = torch.zeros(eval_iters)

        for k in range(eval_iters):
            x, y = get_batch(split)
            logits, loss = model(x, y)
            split_losses[k] = loss.item()

        losses[split] = split_losses.mean()

    model.train()  # set back to training mode
    return losses

# Training Loop
for step in range(max_iters):

    # Evaluate every eval_every steps
    if step % eval_every == 0:
        losses = estimate_loss()
        print(f"Step {step:4d} — Train Loss: {losses['train']:.4f} | Val Loss: {losses['val']:.4f}")

    # Get batch of training data
    x, y = get_batch("train")

    # Forward pass — get predictions and loss
    logits, loss = model(x, y)

    # Backward pass — calculate gradients
    optimizer.zero_grad(set_to_none=True)
    loss.backward()

    # Update weights
    optimizer.step()

# Final evaluation
losses = estimate_loss()
print("-" * 50)
print(f"Training Complete! ✅")
print(f"Final Train Loss — {losses['train']:.4f}")
print(f"Final Val Loss   — {losses['val']:.4f}")

Starting Mini GPT Training on Marathi Text...
Total steps     — 5000
Evaluate every  — 500 steps
Learning rate   — 0.0003
Device          — cuda
--------------------------------------------------
Step    0 — Train Loss: 5.6138 | Val Loss: 5.6125
Step  500 — Train Loss: 2.5760 | Val Loss: 2.5621
Step 1000 — Train Loss: 2.2039 | Val Loss: 2.1770
Step 1500 — Train Loss: 2.0011 | Val Loss: 1.9890
Step 2000 — Train Loss: 1.8867 | Val Loss: 1.8856
Step 2500 — Train Loss: 1.8085 | Val Loss: 1.8173
Step 3000 — Train Loss: 1.7411 | Val Loss: 1.7478
Step 3500 — Train Loss: 1.6905 | Val Loss: 1.7289
Step 4000 — Train Loss: 1.6527 | Val Loss: 1.7020
Step 4500 — Train Loss: 1.6385 | Val Loss: 1.6774
--------------------------------------------------
Training Complete! ✅
Final Train Loss — 1.6030
Final Val Loss   — 1.6478


## Generate Marathi Text

In [17]:
print("Generating Marathi Text with Mini GPT...")
print("=" * 50)

def generate_marathi_text(starting_text, max_tokens=500):
    """
    Generate Marathi text from starting text

    What happens —
    1. Encode starting text to numbers
    2. Feed to model
    3. Model generates next characters one by one
    4. Decode numbers back to Marathi text
    """
    # Encode starting text
    context = torch.tensor(
        encode(starting_text),
        dtype=torch.long,
        device=device
    ).unsqueeze(0)  # add batch dimension

    # Generate text
    with torch.no_grad():
        generated = model.generate(context, max_new_tokens=max_tokens)

    # Decode back to Marathi text
    generated_text = decode(generated[0].tolist())

    return generated_text

# Generate text with different starting texts
print("\nGeneration 1 — Starting with 'मराठी'")
print("-" * 40)
text1 = generate_marathi_text("मराठी", max_tokens=300)
print(text1)

print("\nGeneration 2 — Starting with 'शासन'")
print("-" * 40)
text2 = generate_marathi_text("शासन", max_tokens=300)
print(text2)

print("\nGeneration 3 — Starting with 'महाराष्ट्र'")
print("-" * 40)
text3 = generate_marathi_text("महाराष्ट्र", max_tokens=300)
print(text3)

Generating Marathi Text with Mini GPT...

Generation 1 — Starting with 'मराठी'
----------------------------------------
मराठी घूळण्याचा लताढ आणि नॉची घटक आयुक्तांना पार्पास करत आहे. लसभ्यासंख्याआरण आणि मदत आरोग्य संघाच्या विशिष्टमंडळात नळ आला आहे. या आणि बेकनमाला फायदा परदेशी अकसो, भारतीय तंत्रेकडील आणि अंगर जनपूर रेल्वेचा नुकसानकाल झाला. याप्रकरणी मारला अंटकार नेऊन तो पाहता?
, : स्वतःची स्थापना व्यत्ती नाहीत. आपुरती, सात

Generation 2 — Starting with 'शासन'
----------------------------------------
शासन आणि देशांत आहे.
 भाजपतीला "डी ह्मणत या भीम, त्यानें भारतीय जबाबद्द येणें याचें नये नाही आणि याम पुन्हा त्या बळ्यार्थ पट्ट्यांसमाला घालणाऱ्या सुर्वयंका एकाद्य अंतरिक बाळम फाऊसचे अँटींशन वाघ करण्यात आले आहे. कैसराची पातकाांही गुन्हा पाहण्याबद्दल ख्रेड विरोदर करणवर आखो , किंवा कुडे, रस्ता, करणे, याआधी

Generation 3 — Starting with 'महाराष्ट्र'
----------------------------------------
महाराष्ट्रातील गोष्ट राहिंगातः
हत्यात आलंदांना आंकडून पाहिजे.
नपरदानातील मलाच सामनोरने (४ वान दिवस'.०१- टी 

## Save the Model

In [18]:
import os

# Save model to Google Drive
save_path = '/content/drive/MyDrive/mini_gpt_marathi'
os.makedirs(save_path, exist_ok=True)

# Save model weights
torch.save(
    model.state_dict(),
    f'{save_path}/mini_gpt_marathi.pth'
)

# Save hyperparameters
import json
hyperparams = {
    'vocab_size'    : vocab_size,
    'n_embd'        : n_embd,
    'n_head'        : n_head,
    'n_layer'       : n_layer,
    'block_size'    : block_size,
    'dropout'       : dropout,
    'batch_size'    : batch_size,
    'max_iters'     : max_iters,
    'learning_rate' : learning_rate,
    'final_train_loss' : losses['train'].item(),
    'final_val_loss'   : losses['val'].item()
}

with open(f'{save_path}/hyperparams.json', 'w') as f:
    json.dump(hyperparams, f, indent=4)

print("Model saved to Google Drive ✅")
print(f"Save path — {save_path}")
print(f"\nSaved files —")
print(f"mini_gpt_marathi.pth — model weights")
print(f"hyperparams.json     — model settings")
print(f"marathi_text.txt     — training data")
print(f"vocab.json           — vocabulary")

Model saved to Google Drive ✅
Save path — /content/drive/MyDrive/mini_gpt_marathi

Saved files —
mini_gpt_marathi.pth — model weights
hyperparams.json     — model settings
marathi_text.txt     — training data
vocab.json           — vocabulary


In [19]:
from google.colab import files

# Download model weights
files.download('/content/drive/MyDrive/mini_gpt_marathi/mini_gpt_marathi.pth')

# Download hyperparameters
files.download('/content/drive/MyDrive/mini_gpt_marathi/hyperparams.json')

# Download vocabulary
files.download('/content/drive/MyDrive/mini_gpt_marathi/vocab.json')

print("Files downloading to your laptop ✅")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Files downloading to your laptop ✅
